# Na⁺ and K⁺ currents during a recorded action potential

Copyright (c) 2026 Open Brain Institute

Authors: Ilkan Kiliç

last modified: 08.2026

This notebook analyses precomputed `SingleNeuronSimulation` result entities from the legacy single-cell workflow. It loads existing traces, inspects metadata, selects one voltage/Na⁺/K⁺ set, and plots the result.

In [ ]:
from pathlib import Path

from obi_auth import get_token
from entitysdk.client import Client
from entitysdk.models import SingleNeuronSimulation
from obi_notebook import get_projects
from obi_notebook import get_entities
from obi_notebook.get_environment import get_environment

from analysis_helper import (
    load_simulation_results,
    metadata_units,
    plot_normalized_voltage_and_currents,
    plot_voltage_and_currents,
    plot_zoomed_ap,
    prepare_trace_bundle,
    result_summary,
    select_trace,
    trace_catalog,
)


## Select a project

Authenticate to the platform and select the project that was used to produce the single-cell simulations you want to analyse.

In [ ]:
token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects.get_projects(token)


Enter one `SingleNeuronSimulation` ID to analyse, or leave `simulation_id` empty and select one from the table after running the Python cell below.

Once this is done, the notebook will fetch the selected precomputed result asset.

In [ ]:
client = Client(
    project_context=project_context,
    environment=get_environment(),
    token_manager=token,
)

# Optional: enter one unique SingleNeuronSimulation ID.
simulation_id = ""
if not simulation_id:
    simulation_id_list = get_entities.get_entities("single-neuron-simulation", token, [],
                                                 project_context=project_context,
                                                 multi_select=False,
                                                 page_size=100)
    if not simulation_id_list:
        raise ValueError("Select one SingleNeuronSimulation.")
    simulation_id = simulation_id_list[0]


## Download precomputed results

This will download the selected simulation.

In [ ]:
download_dir = Path("./simulation_results")
download_dir.mkdir(parents=True, exist_ok=True)

simulation = client.get_entity(
    entity_type=SingleNeuronSimulation,
    entity_id=simulation_id,
)
asset = client.download_assets(
    simulation,
    output_path=str(download_dir),
).one()
simulation_path = Path(asset.path).resolve()

print("Result file:")
print(f"  {simulation_path}")


## Parse the result assets

In [ ]:
simulation_result = load_simulation_results([simulation_path])[0]
summary = result_summary([simulation_result])
summary


## Inspect the trace catalog

A result may contain multiple variables, recording locations, stimulus amplitudes, or repetitions. Inspect the catalog before selecting traces.

In [ ]:
selected_data = simulation_result["data"]
catalog = trace_catalog(selected_data)
catalog


## Select one voltage/Na⁺/K⁺ recording

The exact filters below keep all three traces at one recording location and stimulus condition. Change these values using the catalog columns when analysing another result.

In [ ]:
selected_recording = "soma[0]_0.5"
selected_stimulus = "IDREST_0.5"

trace_filters = {
    "voltage": {
        "recording": selected_recording,
        "name": selected_stimulus,
        "variable_name": "v",
    },
    "sodium": {
        "recording": selected_recording,
        "name": selected_stimulus,
        "variable_name": "ina",
    },
    "potassium": {
        "recording": selected_recording,
        "name": selected_stimulus,
        "variable_name": "ik",
    },
}

trace_aliases = {
    "voltage": ("voltage", "membrane_potential", "membrane", "v", "vm"),
    "sodium": ("sodium", "na", "ina", "i_na"),
    "potassium": ("potassium", "k", "ik", "i_k"),
}

selected_traces = {
    name: select_trace(
        selected_data,
        aliases=trace_aliases[name],
        selector=trace_filters[name],
    )
    for name in ("voltage", "sodium", "potassium")
}

selected_trace_metadata = {
    name: trace["metadata"]
    for name, trace in selected_traces.items()
}
selected_trace_metadata


## Prepare the selected traces


In [ ]:
trace_bundle = prepare_trace_bundle(selected_traces)
voltage_units = metadata_units(selected_traces["voltage"], "mV")
current_units = metadata_units(
    selected_traces["sodium"],
    metadata_units(selected_traces["potassium"], "mA/cm²"),
)

print(f"Loaded {len(trace_bundle['t'])} samples from simulation {simulation_id}.")
print(f"Voltage units: {voltage_units}; current units: {current_units}")


## Plot the complete recording

This three-panel view shows membrane voltage, Na⁺ current, and K⁺ current across the complete precomputed recording.

In [ ]:
plot_voltage_and_currents(
    trace_bundle["t"],
    trace_bundle["v"],
    trace_bundle["ina"],
    trace_bundle["ik"],
    voltage_units=voltage_units,
    current_units=current_units,
)


## Compare normalized waveform shapes

This dimensionless view scales voltage from its initial baseline to its peak and each current by its maximum absolute magnitude. The raw plots above preserve physical units.

In [ ]:
plot_normalized_voltage_and_currents(
    trace_bundle["t"],
    trace_bundle["v"],
    trace_bundle["ina"],
    trace_bundle["ik"],
)


## Inspect one action potential

The zoomed view uses a dual y-axis for voltage and ionic currents. With `zoom_window = None`, the helper centers a 10 ms window on the largest voltage peak; set a pair of millisecond bounds to inspect a specific spike.

In [ ]:
zoom_window = None  # For example: (141, 147)
if zoom_window is None:
    zoom_start, zoom_end = None, None
else:
    zoom_start, zoom_end = zoom_window

plot_zoomed_ap(
    trace_bundle["t"],
    trace_bundle["v"],
    trace_bundle["ina"],
    trace_bundle["ik"],
    t_start=zoom_start,
    t_end=zoom_end,
    voltage_units=voltage_units,
    current_units=current_units,
)


## Interpretation

- In the usual current convention, inward Na⁺ current is negative and activates rapidly during depolarization.
- Delayed outward K⁺ current is positive and contributes to repolarization and the return toward the resting voltage.
- The relative timing of the Na⁺ and K⁺ peaks is more informative than their absolute scale when comparing recording locations or parameter sets. Confirm the sign convention and units from the selected metadata.
- A TTX comparison should use a separate precomputed TTX result alongside a separate control result. This notebook only reads and plots supplied assets; it does not modify traces or apply a pharmacological condition.